# Melampo: misura dell'interprete delle immagini (VoxTell + TotalSegmentator)

Solo dati pubblici: Medical Segmentation Decathlon, Task06 Lung (tumori polmonari in TC, licenza CC BY-SA 4.0).

Prima di eseguire, nelle impostazioni del notebook (pannello a destra):
- **Accelerator**: GPU T4 x2 (oppure P100);
- **Internet**: on (serve la verifica del telefono sull'account Kaggle);
- se il repository è privato: **Add-ons → Secrets → GITHUB_TOKEN** con un token GitHub *fine-grained*, solo lettura, solo su `Melampo-AI`.

Poi *Run all*. Alla fine si scarica `/kaggle/working/probe.json` e il log, da portare a Claude.

In [ ]:
# 0. Internet: senza, pip e il download del dataset falliscono ("Temporary failure in name resolution")
import socket
try:
    socket.gethostbyname("pypi.org")
    print("Internet OK")
except OSError:
    raise SystemExit("Internet spento: pannello a destra -> Session options -> Internet: On "
                     "(richiede il telefono verificato in Settings), poi Run All di nuovo.")

In [ ]:
!pip install -q voxtell TotalSegmentator nibabel
# nnInteractive (MIC-DKFZ; pesi CC BY-NC-SA 4.0, come VoxTell): secondo modello per la diagnosi delle misure corte.
# Se l'installazione fallisce il probe gira lo stesso e lo scrive.
!pip install -q nnInteractive || echo "nnInteractive non installato"
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# VoxTell segnala un problema di memoria con PyTorch 2.9.x: in quel caso si scende alla 2.8
import torch, subprocess, sys
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.__version__.startswith("2.9"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "torchvision==0.23.0"], check=True)
    print("installata torch 2.8.0: Run -> Restart & Run All")

In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
REPO = "/kaggle/working/melampo"
if os.path.isdir(os.path.join(REPO, ".git")):
    # già scaricato in un giro precedente: si aggiorna all'ultima versione del ramo principale
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["rm", "-rf", REPO], check=True)   # una cartella lasciata a metà da un giro interrotto
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Medical Segmentation Decathlon (CC BY-SA 4.0) sul registro AWS Open Data (bucket pubblico "msd-for-monai"),
# letto con boto3 senza credenziali, come indica il registro ("--no-sign-request").
# Si scaricano solo i casi che servono. Se l'elenco non trova Task06, la cella stampa cosa c'è nel bucket.
import os, glob, subprocess, sys, tarfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
import boto3
from botocore import UNSIGNED
from botocore.config import Config

N = 30
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
images = sorted(glob.glob("/kaggle/input/**/imagesTr/lung_*.nii*", recursive=True))   # dati aggiunti con "Add Input"

for bucket, region in (("msd-for-monai", "us-west-2"), ("msd-for-monai-eu", "eu-west-2")):
    if images:
        break
    s3 = boto3.client("s3", region_name=region, config=Config(signature_version=UNSIGNED))
    try:
        keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket)
                for o in page.get("Contents", [])]
    except Exception as e:
        print(bucket, "elenco non disponibile:", repr(e)[:300]); continue
    print(bucket, len(keys), "oggetti; esempi:", keys[:15])
    lung = [k for k in keys if "lung" in k.lower()]
    print("con 'lung' nel nome:", lung[:20])
    imgs = sorted(k for k in lung if "imagesTr" in k and k.endswith(".nii.gz") and "/._" not in k)
    if imgs:
        for k in imgs[:N]:
            for kk in (k, k.replace("imagesTr", "labelsTr")):
                dest = os.path.join(DATA, kk)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                if not os.path.exists(dest):
                    s3.download_file(bucket, kk, dest)
    else:
        archive = next((k for k in lung if k.endswith((".tar", ".tar.gz", ".tgz"))), None)
        if archive:
            dest = os.path.join("/kaggle/working", os.path.basename(archive))
            print("scarico l'archivio", archive, "(circa 8,5 GB)")
            s3.download_file(bucket, archive, dest)
            with tarfile.open(dest) as t:
                members = [m for m in t.getmembers() if ("imagesTr/lung_" in m.name or "labelsTr/lung_" in m.name)
                           and "/._" not in m.name]
                wanted = sorted({os.path.basename(m.name) for m in members})[:N]
                t.extractall(DATA, members=[m for m in members if os.path.basename(m.name) in wanted])
            os.remove(dest)
    images = sorted(glob.glob(f"{DATA}/**/imagesTr/lung_*.nii.gz", recursive=True))
print(len(images), "immagini:", [os.path.basename(p) for p in images])
assert images, "nessun dato: mandare a Claude l'output di questa cella"

In [ ]:
import json, os
lobes = ["lung_upper_lobe_left", "lung_lower_lobe_left", "lung_upper_lobe_right", "lung_middle_lobe_right", "lung_lower_lobe_right"]
cases = [{"id": os.path.basename(p).split(".")[0], "image": p, "lesion": p.replace("imagesTr", "labelsTr"),
          "structure": lobes, "prompt": "tumore polmonare", "prompt_en": "lung tumor", "modality": "CT"} for p in images[:N]]
# MSD non ha referti: lo script scrive la frase di un referto con la misura di riferimento (segnata come sintetica)
json.dump(cases, open("/kaggle/working/cases.json", "w"), indent=1)
print(len(cases), "casi")

In [ ]:
!cd /kaggle/working/melampo && python scripts/imaging_probe.py --cases /kaggle/working/cases.json --out /kaggle/working/probe.json --device cuda --fast-organs --diagnose-size --nninteractive 2>&1 | tee /kaggle/working/probe.log

In [ ]:
import json
rows = json.load(open("/kaggle/working/probe.json"))
def mm(x):
    return "-" if x is None else f"{x:.1f}"
for r in rows:
    if "error" in r:
        print(r["id"], "ERRORE:", r["error"]); continue
    rd = r.get("reference_density") or {}
    print(f"{r['id']} ({r['slices']} fette, {r['seconds']:.0f}s) referto{' sintetico' if r['report_is_synthetic'] else ''}: "
          f"{r['report_clause']}  riferimento {mm(r['reference_longest_axial_mm'])} mm, densità rif. {rd.get('composition')} "
          f"mediana {mm(rd.get('median'))}")
    for k in ("prompt", "prompt_en"):
        if k not in r:
            continue
        v, d, s = r[k], r[k].get("density") or {}, r[k].get("second_reading") or {}
        print(f"  {k:9} {v['status']:9} Dice {v['dice']:.2f} (parte principale {v['dice_main_part']:.2f}, "
              f"soglia {mm(v.get('dice_second_reading'))})  misurato {mm(v['measured_longest_axial_mm'])} mm  "
              f"({v.get('measured_on')}, con vetro smerigliato {mm(v.get('with_ground_glass_axial_mm'))} mm, "
              f"Dice {mm(v.get('dice_with_ground_glass'))})  errore {mm(v.get('reference_error_mm'))} mm  "
              f"parti {v['parts']} altre regioni {v['other_regions']}")
        print(f"            seconda lettura {s.get('status')} accordo={s.get('agree')} soglia {mm(s.get('threshold'))} "
              f"{mm(s.get('second_axial_mm'))} mm {list(s.get('notes', []))}  supporto {v['measure_support']}")
        g = v.get("ground_glass_details") or {}
        print(f"            vetro smerigliato: soglia {mm(g.get('threshold'))} polmone {mm(g.get('surrounding'))} "
              f"rumore {mm(g.get('noise'))} voxel aggiunti {g.get('added_voxels')} {g.get('notes')}")
        print(f"            densità {d.get('composition') or d.get('relative')} mediana {mm(d.get('median'))} "
              f"centro->bordo {d.get('core_to_rim')} parte solida {mm(d.get('solid_axial_mm'))} mm {d.get('flags')}")
        print(f"            -> {'A UNA PERSONA: ' + ', '.join(v['review']) if v['needs_review'] else 'accettato'}"
              f"{'  ' + str(v['character_differences']) if v['character_differences'] else ''}")
print()
print("Perché le misure sono corte (solo prompt italiano): errore del diametro rispetto al riferimento, in mm, per fascia")
variants = [("VoxTell", None), ("taglio 0.35", "cut_0.35"), ("taglio 0.2", "cut_0.2"), ("ricampionato 1.5 mm", "resampled_1_5mm"),
            ("nnInteractive", "nninteractive"), ("parti vicine unite", "merged")]
bins = [(0, 30), (30, 50), (50, 999)]
for name, key in variants:
    cells = []
    for lo, hi in bins:
        errs = []
        for r in rows:
            v = r.get("prompt") or {}
            ref, dd = r.get("reference_longest_axial_mm"), v.get("size_diagnostics") or {}
            if not ref or not (lo <= ref < hi):
                continue
            if key is None:
                m = v.get("measured_longest_axial_mm")
            elif key == "merged":
                m = dd.get("merged_near_parts_axial_mm")
            else:
                m = (dd.get(key) or {}).get("axial_mm")
            if m is not None:
                errs.append(m - ref)
        cells.append(f"{lo}-{hi if hi < 999 else '+'} mm: {sum(errs)/len(errs):+.1f} (n={len(errs)})" if errs else f"{lo}-{hi} mm: -")
    print(f"  {name:22} " + "   ".join(cells))
for r in rows:
    dd = (r.get("prompt") or {}).get("size_diagnostics") or {}
    if dd:
        print(f"  {r['id']}: volume {dd.get('volume_ratio', 0):.2f} del riferimento, mancante {dd.get('missed_share_of_reference', 0):.2f} "
              f"(entro 2 mm dal bordo {dd.get('missed_within_2mm_share', 0):.2f}, HU mediana {dd.get('missed_median_hu')}), "
              f"volume morbido {dd.get('soft_volume_ratio')}, errori: {[k for k in dd if k.endswith('error')]}")
        sh = dd.get("shape") or {}
        print(f"      forma: estensioni rif. {sh.get('extent_reference_mm')} regione {sh.get('extent_region_mm')} mm, "
              f"mancante agli estremi dell'asse {sh.get('missed_at_ends_share')}, blob mancanti {sh.get('missed_blobs')} "
              f"(il maggiore {sh.get('missed_largest_blob_share')}, HU {sh.get('missed_largest_blob_median_hu')}), "
              f"sfericità rif. {sh.get('sphericity_reference')} regione {sh.get('sphericity_region')}")
print()
for k in ("prompt", "prompt_en"):
    acc = [r[k] for r in rows if k in r and not r[k]["needs_review"]]
    rev = [r[k] for r in rows if k in r and r[k]["needs_review"]]
    ok = sum(1 for v in acc if v.get("within_strict"))
    print(f"{k}: accettati {len(acc)}, di cui entro max(2 mm, 10%) dal riferimento {ok}; a una persona {len(rev)}, "
          f"di cui davvero fuori {sum(1 for v in rev if v.get('within_strict') is False)}")
print()
print("Copia l'output di questa cella e incollalo nella chat (probe.json è aggiornato dopo ogni caso).")
